# 6.3 Sleep and Heart Rate

**Wearables series, class 3.** In **6.2**, more daily steps went with a much lower rate of new type 2 diabetes. Is the same true for **sleep**, and for **resting heart rate**? We reuse the prospective design from 6.2, measure first, then follow, turned into a **function** so it works for any wearable measure. Sleep comes from **Fitbit**; resting heart rate from **Apple HealthKit**. We finish with a first look at HealthKit **blood glucose**.

### By the end of this notebook you'll be able to

1. Turn an analysis design into a reusable **function**
2. Build valid **nights** of sleep from Fitbit, and baseline resting heart rate from HealthKit's **long** data
3. Recognise a **U-shaped** (non-linear) association, and choose a **reference** group for it
4. Compare results from two data sources of very different sizes, and read **confidence intervals**
5. Describe a new data type (blood glucose) carefully before modelling it

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Question** | Among adults without diabetes, are sleep duration and resting heart rate linked to developing type 2 diabetes? |
| 🧪 **The analysis** | The 6.2 design, as a function: baseline sleep (Fitbit) or resting heart rate (Apple HealthKit), then follow-up for new type 2 diabetes; incidence rates and Cox models. A description of HealthKit blood glucose |
| 🎯 **Why** | Short sleep and a high resting heart rate are both linked to diabetes risk in other studies. Measured before any diagnosis, they avoid reverse causation, as in 6.2 |
| ✅ **What we conclude** | Whether the links appear in *All of Us* wearable users, and their shape |
| ⚠️ **What it can't tell us** | Cause. Sleep and heart rate reflect overall health, stress, work, and fitness. Fitbit and Apple users are different, selected groups |
| 🔗 **Connections** | The design from **6.2**; valid days from **6.1**; HealthKit's long, text-valued data from **6.1**; functions from **3.1**; survival models from **3.4** |

| | |
|---|---|
| 📂 **Uses** | the Controlled Tier dataset (cohort query, Fitbit sleep, BMI); the Apple HealthKit dataset |
| 💾 **Saves** | `sleep_followup.csv`, `rhr_followup.csv` |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Setup, and the design as a function | 10 |
| 3. Sleep (Fitbit) | 25 |
| 4. Resting heart rate (Apple HealthKit) | 20 |
| 5. Blood glucose: a first look | 15 |
| Wrap-up | 10 |

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 95 minutes |
| ⏱️ **Run time** | about **5 minutes** to run every cell |
| 💰 **Data cost** | about **$1**: the cohort query, then sleep ($0.15), steps ($0.01), resting heart rate ($0.39), and glucose ($0.32) in our test run, each priced before it runs |
| 💰 **App cost** | about **$0.27 an hour** while the app is running. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| **Estimated total** | about **$1.50** per student for this class |

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data, or counts of 1 to 20. Glucose readings are especially personal: plot and share only group summaries. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### The same design, a new exposure

```
first valid           end of baseline                          end of follow-up
measurement           (6 months later)                         (last EHR visit, death, or 2025)
    |──── baseline ────|──────────────── follow-up ─────────────────────|
      median sleep, or      does type 2 diabetes appear? when?
      resting heart rate
excluded: anyone with type 2 diabetes before the end of baseline
```

As in **6.2**, the exposure is measured **before** any diagnosis, so it avoids reverse causation by construction.

### Not every link is a straight line

Steps (**6.2**) had a **gradient**: more steps, lower rate. Sleep is often different: both **short** and **long** sleep are linked to worse health, with the lowest risk in the middle. That's a **U shape**. Two consequences:

- A single "per hour of sleep" hazard ratio would **average away** the U, and could show nothing at all.
- The **reference** group should be the middle (here **7 to 8 hours**), so every other group is compared with the lowest-risk one.

### What the devices measure

| Measure | Device | How | Watch out for |
|---|---|---|---|
| **Sleep duration** | Fitbit | movement and heart rate, through the night; one **main sleep** per night (naps are separate) | Fitbits tend to record **less** sleep than people report; only nights the device was worn |
| **Sleep stages** (light, deep, REM) | Fitbit (newer models) | heart rate variability and movement | stage estimates are rough; older devices only record "asleep" and "restless" |
| **Resting heart rate** | Apple Watch | the watch's own daily estimate, at rest | lower with **fitness**, higher with stress, illness, some medicines |
| **Blood glucose** | other devices or apps, shared through Apple Health (e.g. continuous glucose monitors) | depends on the device | who uses one is **not** random: mostly people **already** managing diabetes |

### Reading confidence intervals

Fitbit sleep data covers tens of thousands of people; HealthKit resting heart rate far fewer. Fewer people and fewer new cases mean **wider** confidence intervals. A wide interval that includes 1 means "we can't tell", not "no link".

## Part 2. Setup, and the design as a function

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions
library(survival)
library(broom)

cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id))

cohort %>% count(cohort)

### The 6.2 design, as a function

In 6.2 we applied five rules and built follow-up step by step. Here those steps become one **function** (**3.1**), `build_followup()`. It takes any table with `person_id`, `baseline_end`, and a baseline measure, and returns the people eligible for follow-up, with `event` and `years`. Writing it once means sleep and heart rate get **exactly** the same rules.

In [ ]:
build_followup <- function(baseline, cohort, end_of_data = as.Date("2025-01-01")) {
  baseline %>%
    inner_join(cohort, by = "person_id") %>%
    mutate(age_at_baseline = as.numeric(baseline_end - as.Date(date_of_birth)) / 365.25,
           followup_end    = pmin(as.Date(last_ehr_date), as.Date(death_date), end_of_data, na.rm = TRUE),
           t2d_date        = as.Date(first_t2d_date)) %>%
    filter(cohort != "excluded",                                  # not type 1 or unclear
           !(cohort == "case" & t2d_date <= baseline_end),        # no type 2 diabetes yet
           age_at_baseline >= 18,
           followup_end > baseline_end) %>%                        # some follow-up
    mutate(event     = as.integer(cohort == "case" & t2d_date <= followup_end),
           exit_date = if_else(event == 1, t2d_date, followup_end),
           years     = as.numeric(exit_date - baseline_end) / 365.25,
           age10     = age_at_baseline / 10)
}

# Rates per 1,000 person-years by group, hiding 1 to 20 new cases (and their rates)
rates_by <- function(data, group) {
  out <- data %>%
    group_by({{ group }}) %>%
    summarise(people = n(), n = sum(event), person_years = sum(years)) %>%
    mutate(rate_per_1000_py = round(1000 * n / person_years, 1)) %>%
    hide_small_counts() %>%
    mutate(rate_per_1000_py = if_else(is.na(n), NA_real_, rate_per_1000_py),
           person_years = round(person_years)) %>%
    rename(new_cases = n)
  if (sum(is.na(out$new_cases)) == 1) {
    stop("Only one group is hidden: its count could be worked out from the total and the ",
         "other groups. Combine it with a neighbouring group, then run rates_by() again.", call. = FALSE)
  }
  out
}

**🔍 Read the output:** nothing yet: the functions are defined, ready to use.

- `{{ group }}` (read "curly-curly") passes a **column name** into a dplyr function, so `rates_by(data, sleep_group)` groups by `sleep_group`.
- `rates_by()` also **stops with an error** when exactly one group is hidden: anyone who knows the total number of new cases (which we print) could subtract the other groups and work it out. That's the **derived counts** rule from **3.1**. Stopping, rather than warning, means the table is never printed. The fix: combine that group with its neighbour (as we do in Part 4).
- The rules are exactly those of 6.2's flow table.

The BMI part of the 6.2 query is also reused: a piece of SQL that adds each person's latest BMI in the 2 years before their baseline ends.

In [ ]:
bmi_sql <- str_glue("
  bmi AS (
    SELECT b.person_id,
           ARRAY_AGG(m.value_as_number ORDER BY m.measurement_date DESC LIMIT 1)[OFFSET(0)] AS bmi
    FROM baseline AS b
    JOIN `{CURRENT_DATASET}.measurement` AS m ON m.person_id = b.person_id
    WHERE m.measurement_concept_id = 3038553
      AND m.value_as_number BETWEEN 12 AND 80
      AND m.measurement_date BETWEEN DATE_SUB(b.baseline_end, INTERVAL 2 YEAR) AND b.baseline_end
    GROUP BY b.person_id
  )")

## Part 3. Sleep (Fitbit)

### Valid nights, and baseline sleep

From `sleep_daily_summary`, one row per sleep: we keep the **main sleep** of each night, and **valid nights** of 3 to 16 hours asleep (shorter is likely a nap or a device taken off; longer, an error). Then, as in 6.2, the first 6 months of valid nights, and each person's **median** hours asleep, plus their median share of time in **deep** and **REM** sleep (stages are `NULL` on older devices).

`is_main_sleep` may be stored as `true`/`false` or as text, so we compare its **text** form: that works either way.

💰 Cents: the sleep table is about 6 GB.

In [ ]:
sleep_sql <- str_glue("
WITH nights AS (
  SELECT person_id, sleep_date AS date, minute_asleep,
         SAFE_DIVIDE(minute_deep, minute_asleep) AS deep_share,
         SAFE_DIVIDE(minute_rem,  minute_asleep) AS rem_share
  FROM `{CURRENT_DATASET}.sleep_daily_summary`
  WHERE LOWER(CAST(is_main_sleep AS STRING)) IN ('true', '1', 't')
    AND minute_asleep BETWEEN 180 AND 960
),
first_night AS (
  SELECT person_id, MIN(date) AS first_valid FROM nights GROUP BY person_id
),
baseline AS (
  SELECT n.person_id,
         MIN(n.date) AS baseline_start,
         MAX(n.date) AS baseline_end,
         COUNT(*)    AS baseline_nights,
         APPROX_QUANTILES(n.minute_asleep, 100)[OFFSET(50)] / 60 AS sleep_hours,
         APPROX_QUANTILES(n.deep_share, 100)[OFFSET(50)] AS deep_share,
         APPROX_QUANTILES(n.rem_share, 100)[OFFSET(50)]  AS rem_share
  FROM nights AS n
  JOIN first_night AS f ON n.person_id = f.person_id
  WHERE n.date < DATE_ADD(f.first_valid, INTERVAL 6 MONTH)
  GROUP BY n.person_id
),
{bmi_sql}
SELECT b.*, bmi.bmi
FROM baseline AS b LEFT JOIN bmi ON b.person_id = bmi.person_id
WHERE b.baseline_nights >= 10")

sleep_baseline <- run_if_cheap(sleep_sql, max_dollars = 2) %>%
  mutate(person_id = as.character(person_id))

last_query_cost()

sleep_baseline %>%
  summarise(people = n(),
            median_nights = median(baseline_nights),
            median_sleep_hours = round(median(sleep_hours), 2),
            pct_with_stages = round(100 * mean(!is.na(rem_share)), 1),
            median_rem_pct  = round(100 * median(rem_share, na.rm = TRUE), 1),
            median_deep_pct = round(100 * median(deep_share, na.rm = TRUE), 1))

**🔍 Read the output:** how many Fitbit users have a sleep baseline (10+ valid nights in their first 6 months), their typical hours asleep, and how many have sleep **stages** (newer devices only). Healthy adults typically spend about 20 to 25% of sleep in REM and 15 to 20% in deep sleep.

Fitbit's "asleep" excludes time lying awake, so it's usually **less** than the hours people report sleeping in surveys.

In our test run: **54,158** people, a median of 124 valid nights each, and a median of **6.7 hours** asleep. 79% had sleep stages, with a median of 20% REM and 15% deep sleep, both in the usual healthy range.

### Follow-up and sleep groups

In [ ]:
sleep_followup <- build_followup(sleep_baseline, cohort) %>%
  mutate(sleep_group = cut(sleep_hours, breaks = c(0, 6, 7, 8, Inf), right = FALSE,
                           labels = c("under 6 hours", "6 to 7 hours", "7 to 8 hours", "8 hours or more")),
         sleep_group = relevel(sleep_group, ref = "7 to 8 hours"))   # the middle group is the reference

sleep_followup %>%
  summarise(people = n(), new_cases = sum(event), person_years = round(sum(years)),
            median_followup_years = round(median(years), 1))

rates_by(sleep_followup, sleep_group)

**🔍 Read the output:** the people followed, new cases, and person-years, then the **incidence rate** by sleep group. Look at the **shape**: is the rate lowest in the middle (a U), or does it fall steadily with more sleep?

In our test run: **24,507** people followed, **1,251** new cases over 91,514 person-years (median 2.9 years). Rates per 1,000 person-years: **21.5** under 6 hours, 13.2 for 6 to 7, **10.8** for 7 to 8, and 11.6 for 8 or more.

`relevel(…, ref = "7 to 8 hours")` makes the middle group the **reference** for the Cox model, which is why it's listed first.

### A picture of the shape

In [ ]:
rates_by(sleep_followup, sleep_group) %>%
  mutate(sleep_group = factor(sleep_group, levels = c("under 6 hours", "6 to 7 hours",
                                                      "7 to 8 hours", "8 hours or more"))) %>%
  ggplot(aes(x = sleep_group, y = rate_per_1000_py)) +
  geom_col(fill = "#3182bd", width = 0.6) +
  geom_text(aes(label = rate_per_1000_py), vjust = -0.5, size = 4.5) +
  labs(x = "Hours asleep a night (Fitbit, baseline median)",
       y = "New type 2 diabetes per 1,000 person-years",
       title = "Sleep and new type 2 diabetes", subtitle = "Unadjusted rates") +
  theme_minimal(base_size = 14)

**🔍 Read the output:** one bar per sleep group. A U shape has its lowest bar in the middle; a gradient falls (or rises) steadily. These rates are **unadjusted**: people who sleep less may also be younger, or work more.

In our test run, the rate was **twice as high** with under 6 hours as with 7 to 8, and only slightly higher with 8 hours or more: mostly a **short-sleep** effect, with at most a small upturn for long sleep.

### Cox models

As in 6.2: age and sex, then + BMI, on the same people. And we add **daily steps** in a third model: people who sleep well may also walk more.

In [ ]:
steps_baseline <- run_if_cheap(str_glue("
  SELECT person_id, APPROX_QUANTILES(steps, 100)[OFFSET(50)] AS steps
  FROM `{CURRENT_DATASET}.activity_summary`
  WHERE steps BETWEEN 100 AND 45000
  GROUP BY person_id"), max_dollars = 1) %>%
  mutate(person_id = as.character(person_id), steps_1000 = steps / 1000)

sleep_cox_data <- sleep_followup %>%
  left_join(steps_baseline, by = "person_id") %>%
  filter(sex_at_birth %in% c("Female", "Male"), !is.na(bmi), !is.na(steps_1000)) %>%
  mutate(sex_at_birth = factor(sex_at_birth), bmi5 = bmi / 5)

c(people = nrow(sleep_cox_data), new_cases = sum(sleep_cox_data$event))

s1 <- coxph(Surv(years, event) ~ sleep_group + age10 + sex_at_birth, data = sleep_cox_data)
s2 <- coxph(Surv(years, event) ~ sleep_group + age10 + sex_at_birth + bmi5, data = sleep_cox_data)
s3 <- coxph(Surv(years, event) ~ sleep_group + age10 + sex_at_birth + bmi5 + steps_1000, data = sleep_cox_data)

bind_rows(
  tidy(s1, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "age and sex"),
  tidy(s2, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "+ BMI"),
  tidy(s3, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "+ BMI + steps")) %>%
  filter(str_starts(term, "sleep_group")) %>%
  transmute(term = str_remove(term, "sleep_group"), model,
            HR = sprintf("%.2f (%.2f to %.2f)", estimate, conf.low, conf.high)) %>%
  pivot_wider(names_from = model, values_from = HR)

**🔍 Read the output:** hazard ratios for each sleep group compared with **7 to 8 hours**, in three models.

In our test run (15,712 people with BMI and steps; 591 new cases):

| Compared with 7 to 8 hours | Age and sex | + BMI | + BMI + steps |
|---|---|---|---|
| under 6 hours | **1.97** (1.59 to 2.44) | 1.48 (1.19 to 1.84) | **1.46** (1.18 to 1.81) |
| 6 to 7 hours | 1.18 (0.96 to 1.44) | 1.09 (0.89 to 1.33) | 1.13 (0.92 to 1.38) |
| 8 hours or more | 0.93 (0.65 to 1.34) | 0.98 (0.68 to 1.40) | 0.90 (0.63 to 1.29) |

- **Short sleep** went with about **twice** the rate of new type 2 diabetes. BMI explained part of it (1.97 → 1.48), steps almost none (1.48 → 1.46).
- **Long sleep** wasn't linked: no U shape here. Other studies, mostly using **self-reported** sleep, often find one. Fitbit measures sleep differently, and few people slept 8+ hours by its measure. We expected a U; **the data decides**.
- HRs **above 1** for both short and long sleep would have been a U shape.
- The steps used here are each person's median over **all** their Fitbit days, a simplification: for a paper, use steps over the **same** baseline months.

### ✍️ Your turn

What if we'd treated sleep as a **number** (hours, per 1 hour) instead of groups? Fit `coxph(Surv(years, event) ~ sleep_hours + age10 + sex_at_birth, data = sleep_cox_data)`. Does the single hazard ratio describe the groups' results well?

<details>
<summary><b>Show answer</b></summary>

```r
coxph(Surv(years, event) ~ sleep_hours + age10 + sex_at_birth, data = sleep_cox_data) %>%
  tidy(exponentiate = TRUE, conf.int = TRUE)
```

If the groups show a U, a single "per hour" hazard ratio averages the two sides and can come out close to 1: it **hides** the shape. When a link might not be a straight line, look at groups (or a spline) before summarising it with one number.

</details>

In [ ]:
# Your code here

## Part 4. Resting heart rate (Apple HealthKit)

Fitbit's tables in CDR v9 have heart-rate **zones**, but no resting heart rate. Apple HealthKit has it, as `RESTING_HEART_RATE` in `vital_measurements`: one of the **long**-format tables from **6.1**, with `value` stored as **text**.

The same steps: daily values (the average if a day has several), plausible values only (30 to 150 beats per minute), the first 6 months, each person's **median**, 10 or more days. Plus BMI from the **main** dataset: a query can join tables from two datasets, as long as you can read both.

💰 About $0.30: the HealthKit table isn't clustered by person.

In [ ]:
rhr_sql <- str_glue("
WITH days AS (
  SELECT person_id,
         DATE(SAFE_CAST(datetime AS TIMESTAMP)) AS date,
         AVG(SAFE_CAST(value AS FLOAT64)) AS rhr
  FROM `{HEALTHKIT_DATASET}.vital_measurements`
  WHERE activity_type = 'RESTING_HEART_RATE'
  GROUP BY 1, 2
  HAVING rhr BETWEEN 30 AND 150
),
first_day AS (
  SELECT person_id, MIN(date) AS first_valid FROM days GROUP BY person_id
),
baseline AS (
  SELECT d.person_id,
         MIN(d.date) AS baseline_start,
         MAX(d.date) AS baseline_end,
         COUNT(*)    AS baseline_days,
         APPROX_QUANTILES(d.rhr, 100)[OFFSET(50)] AS rhr
  FROM days AS d
  JOIN first_day AS f ON d.person_id = f.person_id
  WHERE d.date < DATE_ADD(f.first_valid, INTERVAL 6 MONTH)
  GROUP BY d.person_id
),
{bmi_sql}
SELECT b.*, bmi.bmi
FROM baseline AS b LEFT JOIN bmi ON b.person_id = bmi.person_id
WHERE b.baseline_days >= 10")

rhr_baseline <- run_if_cheap(rhr_sql, max_dollars = 2) %>%
  mutate(person_id = as.character(person_id))

last_query_cost()

rhr_followup <- build_followup(rhr_baseline, cohort) %>%
  mutate(rhr_group = cut(rhr, breaks = c(0, 60, 70, Inf), right = FALSE,      # 70+ combined:
                         labels = c("under 60", "60 to 69", "70 or more")),     # 80+ alone is too small
         rhr10 = rhr / 10)

c(baseline_people = nrow(rhr_baseline), followed = nrow(rhr_followup), new_cases = sum(rhr_followup$event))

rates_by(rhr_followup, rhr_group)

**🔍 Read the output:** the cost; how many HealthKit users have a resting-heart-rate baseline and how many could be followed; and rates by resting heart rate. In our test run: **11,258** people with a baseline, **5,306** followed, and **133** new cases: far fewer than with Fitbit sleep.

We use **three** groups, not four: with four, the highest group (80 or more) had too few new cases to show, and its count could have been worked out from the total (`rates_by()` stops when that happens).

Fewer people also means a **shorter** history: Apple Health data in *All of Us* is newer.

### Cox model: per 10 beats per minute

With few new cases, four groups would give very wide intervals. One number, **per 10 beats per minute**, uses all the information (heart rate is usually close to a straight-line relationship):

In [ ]:
rhr_cox_data <- rhr_followup %>%
  filter(sex_at_birth %in% c("Female", "Male")) %>%
  mutate(sex_at_birth = factor(sex_at_birth), bmi5 = bmi / 5)

r1 <- coxph(Surv(years, event) ~ rhr10 + age10 + sex_at_birth, data = rhr_cox_data)
r2 <- coxph(Surv(years, event) ~ rhr10 + age10 + sex_at_birth + bmi5,
            data = rhr_cox_data %>% filter(!is.na(bmi)))

bind_rows(
  tidy(r1, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "age and sex", people = r1$n, new_cases = r1$nevent),
  tidy(r2, exponentiate = TRUE, conf.int = TRUE) %>% mutate(model = "+ BMI", people = r2$n, new_cases = r2$nevent)) %>%
  filter(term == "rhr10") %>%
  transmute(model, people, new_cases,
            HR_per_10_bpm = sprintf("%.2f (%.2f to %.2f)", estimate, conf.low, conf.high)) %>%
  mutate(new_cases = if_else(new_cases <= 20, NA_real_, new_cases))

**🔍 Read the output:** the hazard ratio for type 2 diabetes for every **10 beats per minute** higher resting heart rate, with and without BMI, and how many people and new cases each model used. Notice the **width** of the interval compared with the Fitbit models: that's what fewer people and fewer events do.

`r1$n` and `r1$nevent` are the people and events the model actually used (after dropping missing values).

In our test run: **HR 1.44 per 10 bpm** (1.19 to 1.75), adjusted for age and sex (5,289 people, 133 new cases); **1.25** (1.00 to 1.55) with BMI. A higher resting heart rate went with more new diabetes, but the interval is **much wider** than for sleep, and with BMI it barely excludes 1.

**💬 Discuss:** resting heart rate is lower in people who are **fit**. Is a link with diabetes about the heart, about fitness, or about something else (stress, illness, medicines like beta-blockers that lower heart rate)?

## Part 5. Blood glucose: a first look

About 1,000 *All of Us* participants share **blood glucose** readings through Apple Health, often from **continuous glucose monitors** (CGMs) that read every 5 minutes. Before modelling a new data type, **describe** it: who has it, how often it's measured, and what it looks like.

For each person, in BigQuery: the number of readings and days, readings per day, the median, and the share of readings **in range** (70 to 180 mg/dL, the usual CGM target) and **above 180**.

💰 About $0.30.

In [ ]:
glucose <- run_if_cheap(str_glue("
  SELECT person_id,
         COUNT(*) AS readings,
         COUNT(DISTINCT DATE(SAFE_CAST(datetime AS TIMESTAMP))) AS days,
         APPROX_QUANTILES(SAFE_CAST(value AS FLOAT64), 100)[OFFSET(50)] AS median_glucose,
         AVG(IF(SAFE_CAST(value AS FLOAT64) BETWEEN 70 AND 180, 1, 0)) AS share_in_range,
         AVG(IF(SAFE_CAST(value AS FLOAT64) > 180, 1, 0)) AS share_above_180
  FROM `{HEALTHKIT_DATASET}.vital_measurements`
  WHERE activity_type = 'BLOOD_GLUCOSE' AND units = 'mg/dL'
  GROUP BY person_id"), max_dollars = 2) %>%
  mutate(person_id = as.character(person_id),
         readings_per_day = readings / days,
         likely_cgm = readings_per_day >= 24)    # a CGM gives up to 288 readings a day

last_query_cost()

glucose %>%
  left_join(cohort %>% select(person_id, cohort), by = "person_id") %>%
  mutate(cohort = replace_na(cohort, "not in the cohort query")) %>%
  group_by(cohort) %>%
  summarise(n = n(),
            pct_likely_cgm = round(100 * mean(likely_cgm), 1),
            median_days = median(days),
            median_glucose = round(median(median_glucose)),
            median_pct_in_range = round(100 * median(share_in_range), 1),
            median_pct_above_180 = round(100 * median(share_above_180), 1)) %>%
  hide_small_counts() %>%
  mutate(across(-c(cohort, n), ~ if_else(is.na(n), NA_real_, as.numeric(.x)))) %>%
  rename(people = n)

**🔍 Read the output:** for cases, controls, excluded people (type 1 or unclear diabetes), and people not in the cohort query (no EHR): how many have glucose data, the share whose readings look like a **CGM**, how many days they recorded, and their typical glucose and **time in range**. Groups of 20 or fewer people are hidden.

In our test run:

| | People | Likely CGM | Median days | Median glucose | In range (70–180) |
|---|---|---|---|---|---|
| Cases (type 2) | 340 | 30% | 45 | 129 | 92% |
| Controls | 303 | 26% | **6** | 103 | 100% |
| Excluded (type 1 or unclear) | 114 | **83%** | 104 | 144 | **74%** |
| Not in the cohort query | 263 | 33% | 23 | 120 | 97% |

- **Excluded** people, many with **type 1** diabetes, mostly use **CGMs**, record for months, and spend the least time in range.
- **Cases** record for weeks, mostly with occasional readings.
- **Controls** have glucose data for only a **few days**, perhaps short trials of over-the-counter CGMs, or a one-off test.
- So **who** has glucose data, and **how much**, depends on whether they have diabetes: **selection by the outcome**. Glucose data can't be used like steps to predict who **develops** diabetes. What it **can** show is how people with diabetes manage their glucose: a question for diabetes **management**, not prevention.

## ⚠️ Common errors

| Message or problem | Means | Fix |
|---|---|---|
| *No matching signature for aggregate function … STRING* | a HealthKit `value` is text | `SAFE_CAST(value AS FLOAT64)` |
| `is_main_sleep` matches nothing | it's stored differently than expected | `SELECT DISTINCT CAST(is_main_sleep AS STRING)` to see its values |
| A rates table is mostly `NA` | groups with 1 to 20 new cases are hidden | combine groups, or use a single per-unit model (Part 4) |
| *Only one group is hidden …* (an error from `rates_by()`) | that group's count could be worked out from the total | combine it with a neighbouring group, and run again |
| Very wide confidence intervals | few people or few new cases | expected with HealthKit; say so, and don't over-interpret |
| *object 'build_followup' not found* | Part 2 wasn't run | run Part 2 first |

## ✅ Key takeaways

- A design written as a **function** applies the **same rules** to every exposure
- Not every link is a straight line: look at **groups** (and choose a sensible **reference**) before using one number, and let the data decide the shape (here, short sleep, not a U)
- Smaller data sources give **wider** confidence intervals: "can't tell" isn't "no link"
- Fitbit and Apple users are **different** selected groups; results from each describe them
- **Describe** a new data type before modelling it: glucose users are selected **by the outcome**

**This is the end of the wearables series.** Next: the **machine learning series** (7.x) uses steps, sleep, surveys, and more to **predict** who develops type 2 diabetes.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).

## 🧑‍🏫 Instructor notes

- **The function** is the bridge from 6.2: show that it's the 6.2 flow table, written once.
- **Ask for predictions** before showing the sleep rates: straight line or U? Then show the "per hour" model in the exercise, and how it can hide a U.
- **Resting heart rate** is a good place to discuss **power**: the same question with far fewer people gives a much less certain answer.
- **Glucose** is the best example in the course of **selection by the outcome**: the people with the data are the people with the disease.
- **Privacy:** glucose traces are very personal. Only ever show group summaries outside the Workbench.